In [21]:
!pip install fair-esm pandas biopython torch tqdm --quiet

import torch
import esm
import pandas as pd
from itertools import combinations
from tqdm import tqdm

In [22]:
exons = {
1: "MKLRRLQKALCLDLLSLSAACDALDQHNLKQNDQPMDILQIINCLTTIYDRLEQEHNNLVNVPLCVDMCLNWLLNVYDTGRTGRIRVLSFKTGIISLCKAHLEDKYRYLFKQVASSTGFCDQRRLGLLLHDSIQIPRQLGEVASFGGSNIEPSVRSCFQFAMASETVDFRGEDSAKEGKEISICSQNPTVTKLLGGKGVCSGLAKEEINLRPWKSFVNKKMTFLGNILLVKTADENGQEEEINLFKNLWANNYSTTEVANNKPEIEAALFLDWMRLEPQSMVWLPVLHRVAAAETAKHQAKCNICKECPIIGFRYRSLKHFNYDICQSCFFSGRVAKGHKMHYPMVEYCTPTTSGEDVRDFAKVLKNKFRTKRYFAKHPRMGYLPVQTVLEGDNMETPVTLINFWPVDSAPASSPQLSHDECPSLLLLEFVLS",
45: "MKLRRLQKALCLDLLSLSAACDALDQHNLKQNDQPMDILQIINCLTTIYDRLEQEHNNLVNVPLCVDMCL",
46: "MKLRRLQKALCLDLLSLSAACDALDQHNLKQNDQPMDILQIINCLTTIYDRLEQEHNNLVNVPLCVDMCLNWLLNVYDTGRTGRIRVLSFKTGIISLCKAHLEDKYRYLFKQVASSTGFCDQRRLGLLLHDSIQIPRQLG",
47: "MKLRRLQKALCLDLLSLSAACDALDQHNLKQNDQPMDILQIINCLTTIYDRLEQEHNNLVNVPLCVDMCLEVASFGGSNIEPSVRSCFQFAMASETVDFRGEDSAKEGKEISICSQNPTVTKLLGGKGVCSGLAKEEINL",
48: "MKLRRLQKALCLDLLSLSAACDALDQHNLKQNDQPMDILQIINCLTTIYDRLEQEHNNLVNVPLCVDMCLRPWKSFVNKKMTFLGNILLVKTADENGQEEEINLFKNLWANNYSTTEVANNKPEIEAALFLDWMRLEPQS",
49: "MKLRRLQKALCLDLLSLSAACDALDQHNLKQNDQPMDILQIINCLTTIYDRLEQEHNNLVNVPLCVDMCLMVWLPVLHRVAAAETAKHQAKCNICKECPIIGFRYRSLKHFNYDICQSCFFSGRVAKGHKMHYPMVEYCT",
50: "MKLRRLQKALCLDLLSLSAACDALDQHNLKQNDQPMDILQIINCLTTIYDRLEQEHNNLVNVPLCVDMCLPTTSGEDVRDFAKVLKNKFRTKRYFAKHPRMGYLPVQTVLEGDNMETPVTLINFWPVDSAPASSPQLSHD",
}

In [23]:
# Example: skip any single exon, or any combination you want
variants = []
variant_names = []

# Here we skip 1 exon at a time for simplicity; you can extend to multiple skips
for skip_exon in exons.keys():
    seq = "".join([exons[e] for e in exons.keys() if e != skip_exon])
    variants.append(seq)
    variant_names.append(f"skip_exon_{skip_exon}")


In [24]:
print(variant_names)

['skip_exon_1', 'skip_exon_45', 'skip_exon_46', 'skip_exon_47', 'skip_exon_48', 'skip_exon_49', 'skip_exon_50']


In [25]:
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
model.eval()
batch_converter = alphabet.get_batch_converter()


In [26]:
# 5️⃣ Compute embeddings for all variants
layer_idx = model.num_layers

sequence_representations = []
for name, seq in tqdm(zip(variant_names, variants), total=len(variants)):
    batch_labels, batch_strs, batch_tokens = batch_converter([(name, seq)])
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[layer_idx], return_contacts=False)
    token_representations = results["representations"][layer_idx]
    # per-sequence representation: mean of all residue embeddings
    seq_len = (batch_tokens != alphabet.padding_idx).sum(1)
    seq_repr = token_representations[0, 1:seq_len-1].mean(0)
    sequence_representations.append(seq_repr)

100%|██████████| 7/7 [00:16<00:00,  2.32s/it]


In [27]:
# 6️⃣ Convert embeddings to dataframe for easy scoring
emb_matrix = torch.stack(sequence_representations)
df_embeddings = pd.DataFrame(emb_matrix.numpy(), index=variant_names)

In [28]:
print(exons[1])

MKLRRLQKALCLDLLSLSAACDALDQHNLKQNDQPMDILQIINCLTTIYDRLEQEHNNLVNVPLCVDMCLNWLLNVYDTGRTGRIRVLSFKTGIISLCKAHLEDKYRYLFKQVASSTGFCDQRRLGLLLHDSIQIPRQLGEVASFGGSNIEPSVRSCFQFAMASETVDFRGEDSAKEGKEISICSQNPTVTKLLGGKGVCSGLAKEEINLRPWKSFVNKKMTFLGNILLVKTADENGQEEEINLFKNLWANNYSTTEVANNKPEIEAALFLDWMRLEPQSMVWLPVLHRVAAAETAKHQAKCNICKECPIIGFRYRSLKHFNYDICQSCFFSGRVAKGHKMHYPMVEYCTPTTSGEDVRDFAKVLKNKFRTKRYFAKHPRMGYLPVQTVLEGDNMETPVTLINFWPVDSAPASSPQLSHDECPSLLLLEFVLS


In [29]:
wt_seq = exons[1] # "".join([exons[e] for e in sorted(exons.keys())])
batch_labels, batch_strs, batch_tokens = batch_converter([("WT", wt_seq)])
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[layer_idx])
wt_repr = results["representations"][layer_idx][0, 1:batch_tokens.shape[1]-1].mean(0)

from torch.nn.functional import cosine_similarity
scores = []
for vec in sequence_representations:
    scores.append(cosine_similarity(vec, wt_repr, dim=0).item())

df_embeddings["similarity_to_WT"] = scores

In [30]:
# 8️⃣ Rank variants
df_ranked = df_embeddings.sort_values("similarity_to_WT", ascending=False)
print(df_ranked)

# 9️⃣ Save results for downstream AlphaFold / structure modeling
df_ranked.to_csv("dmd_exon_skipped_scores.csv")
print("Saved scoring table as dmd_exon_skipped_scores.csv")

                     0         1         2         3         4         5  \
skip_exon_1  -0.195200 -0.517205  0.092285  0.288426 -0.221582 -0.090271   
skip_exon_49 -0.253561 -0.598115  0.181114  0.279261 -0.231466  0.025287   
skip_exon_48 -0.255492 -0.584726  0.170921  0.286044 -0.229165  0.019402   
skip_exon_47 -0.270882 -0.575000  0.172529  0.310952 -0.227819  0.004909   
skip_exon_50 -0.273035 -0.635227  0.219550  0.311545 -0.252073  0.007575   
skip_exon_46 -0.278064 -0.555581  0.197008  0.282018 -0.224967  0.038973   
skip_exon_45 -0.287290 -0.581910  0.196370  0.289678 -0.222167  0.049648   

                     6         7         8         9  ...       311       312  \
skip_exon_1  -0.012131 -0.024769 -0.016571  0.077870  ...  0.081593 -0.244213   
skip_exon_49  0.157517 -0.011380 -0.077859  0.056710  ...  0.147199 -0.210460   
skip_exon_48  0.120531 -0.022485 -0.059798  0.071505  ...  0.140954 -0.233419   
skip_exon_47  0.126341 -0.041295 -0.050692  0.081603  ...  0.138069